# Support Vector Machines (SVM) — Demo Notebook

This notebook serves as an introduction to SVM:
- hyperplanes and margins
- soft-margin SVM and the role of **C**
- the kernel trick and common kernels (linear / polynomial / RBF)
- how **kernel choices + parameters** change the decision boundary

We'll focus on **visual intuition** using small 2D datasets so we can *see* the decision boundary.

---

## Learning goals
By the end, students should be able to:
1. Fit a linear SVM and interpret **support vectors** and **margins**.
2. Explain how **C** changes the margin/overlap tradeoff (soft margin).
3. Compare kernels (linear vs polynomial vs RBF) on non-linear data.
4. Describe how **γ (gamma)** affects the RBF decision boundary.

## Setup

We will use scikit-learn’s `SVC` (Support Vector Classifier).
A typical best practice is to **standardize** features before fitting an SVM
(especially for RBF / polynomial kernels).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(7)

## Helper functions: plotting decision boundaries, margins, and support vectors

We’ll draw:
- a background "decision region" (predicted class on a grid)
- the decision boundary (where decision function = 0)
- margins (decision function = ±1) for **linear SVMs**
- the support vectors (the points that sit on/inside the margin)

In [ ]:
def plot_decision_regions(ax, model, X, y, padding=0.6, grid_step=0.02, alpha=0.25):
    # Build a mesh
    x_min, x_max = X[:, 0].min() - padding, X[:, 0].max() + padding
    y_min, y_max = X[:, 1].min() - padding, X[:, 1].max() + padding
    xx, yy = np.meshgrid(np.arange(x_min, x_max, grid_step),
                         np.arange(y_min, y_max, grid_step))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # Predict on grid
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=alpha)

    # Scatter points
    ax.scatter(X[:, 0], X[:, 1], c=y, s=30)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

def plot_decision_contours(ax, model, X, levels=(0,), grid_step=0.02):
    x_min, x_max = X[:, 0].min() - 0.8, X[:, 0].max() + 0.8
    y_min, y_max = X[:, 1].min() - 0.8, X[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.arange(x_min, x_max, grid_step),
                         np.arange(y_min, y_max, grid_step))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # decision_function gives signed distance-like scores (scale depends on model)
    df = model.decision_function(grid).reshape(xx.shape)
    ax.contour(xx, yy, df, levels=levels)

def highlight_support_vectors(ax, pipeline):
    # Extract the scaler and SVC from the pipeline
    scaler = pipeline.named_steps["scaler"]
    svc = pipeline.named_steps["svc"]
    
    # Get support vectors in scaled space
    sv_scaled = svc.support_vectors_
    
    # Transform back to original feature space
    sv = scaler.inverse_transform(sv_scaled)
    
    ax.scatter(sv[:, 0], sv[:, 1], s=140, facecolors='none', edgecolors='k', linewidths=1.5)

## 1) Linear SVM on linearly separable data

Recall the idea: find a separating hyperplane with the **largest margin** ("widest possible street"),
and the boundary is determined by only a handful of points called **support vectors**.

In [ ]:
# Linearly separable-ish blobs
X, y = make_blobs(n_samples=120, centers=2, cluster_std=1.0, random_state=7)

# Linear SVM. Large C approximates a hard-margin when data is separable.
linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="linear", C=1e3))
])
linear_svm.fit(X, y)

fig, ax = plt.subplots(figsize=(6, 5))
plot_decision_regions(ax, linear_svm, X, y)
plot_decision_contours(ax, linear_svm, X, levels=(0,))
# Margins for linear SVM: decision_function = ±1
plot_decision_contours(ax, linear_svm, X, levels=(-1, 1))
highlight_support_vectors(ax, linear_svm)
ax.set_title("Linear SVM: decision boundary (0) and margins (±1)")
plt.show()

print("Number of support vectors:", linear_svm.named_steps["svc"].support_.shape[0])

## 2) Soft margin and the role of **C**

When perfect separation is impossible, we allow some points to enter the margin (or even be misclassified).
The hyperparameter **C controls**:
- how much "spill into the street" we allow
- the tradeoff between a wider margin vs fewer violations

We’ll visualize the effect of different C values.

In [ ]:
# Overlapping blobs (hard separation is difficult)
X, y = make_blobs(n_samples=160, centers=2, cluster_std=2.2, random_state=7)

Cs = [0.01, 1, 10]
fig, axes = plt.subplots(1, len(Cs), figsize=(15, 4), sharex=True, sharey=True)

for ax, C in zip(axes, Cs):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="linear", C=C))
    ])
    model.fit(X, y)

    plot_decision_regions(ax, model, X, y)
    plot_decision_contours(ax, model, X, levels=(-1, 0, 1))
    highlight_support_vectors(ax, model)
    ax.set_title(f"Linear SVM (C={C})\n#SV={model.named_steps['svc'].support_.shape[0]}")

plt.suptitle("Soft margin SVM: how C changes the margin and support vectors", y=1.05)
plt.tight_layout()
plt.show()

### Interpretation
- **Small C**: prioritizes a **wider margin** (more violations allowed) → often **more** support vectors.
- **Large C**: prioritizes fitting training data (fewer violations) → narrower margin, can overfit.

This matches the “street width” intuition from the slides.

## 3) When linear boundaries fail: non-linear data

Now let’s use a classic non-linear dataset: **two moons**.
We'll compare:
- linear kernel
- polynomial kernel
- RBF (Gaussian) kernel

In [ ]:
X, y = make_moons(n_samples=220, noise=0.2, random_state=7)

models = [
    ("Linear", Pipeline([("scaler", StandardScaler()),
                         ("svc", SVC(kernel="linear", C=1.0))])),
    ("Poly (deg=3)", Pipeline([("scaler", StandardScaler()),
                               ("svc", SVC(kernel="poly", degree=3, C=1.0, gamma="scale"))])),
    ("RBF", Pipeline([("scaler", StandardScaler()),
                      ("svc", SVC(kernel="rbf", C=1.0, gamma="scale"))])),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, models):
    model.fit(X, y)
    plot_decision_regions(ax, model, X, y)
    plot_decision_contours(ax, model, X, levels=(0,))
    highlight_support_vectors(ax, model)
    ax.set_title(f"{name}\n#SV={model.named_steps['svc'].support_.shape[0]}")
plt.suptitle("Kernel choice changes the shape of the decision boundary", y=1.05)
plt.tight_layout()
plt.show()

## 4) RBF kernel: the role of **γ (gamma)**

For the RBF kernel, gamma controls how quickly similarity falls off with distance.
Intuition:
- **Small γ** → smoother boundary (can underfit)
- **Large γ** → very wiggly boundary (can overfit)

Let’s sweep gamma values to see the boundary change.

In [ ]:
X, y = make_moons(n_samples=220, noise=0.2, random_state=7)

gammas = [0.1, 0.5, 5]
C = 1.0

fig, axes = plt.subplots(1, len(gammas), figsize=(15, 4), sharex=True, sharey=True)
for ax, g in zip(axes, gammas):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="rbf", C=C, gamma=g))
    ])
    model.fit(X, y)

    plot_decision_regions(ax, model, X, y)
    plot_decision_contours(ax, model, X, levels=(0,))
    highlight_support_vectors(ax, model)
    ax.set_title(f"RBF (gamma={g}, C={C})\n#SV={model.named_steps['svc'].support_.shape[0]}")
plt.suptitle("RBF kernel: increasing gamma makes the boundary more flexible", y=1.05)
plt.tight_layout()
plt.show()

## 5) Polynomial kernel: degree and the “wiggliness” of the boundary

Polynomial kernels can produce increasingly complex boundaries as degree increases.
We’ll hold C fixed and vary the degree.

In [ ]:
X, y = make_moons(n_samples=220, noise=0.2, random_state=7)

degrees = [2, 3, 5]
C = 1.0

fig, axes = plt.subplots(1, len(degrees), figsize=(15, 4), sharex=True, sharey=True)
for ax, d in zip(axes, degrees):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="poly", degree=d, C=C, gamma="scale", coef0=1.0))
    ])
    model.fit(X, y)

    plot_decision_regions(ax, model, X, y)
    plot_decision_contours(ax, model, X, levels=(0,))
    highlight_support_vectors(ax, model)
    ax.set_title(f"Poly (degree={d}, C={C})\n#SV={model.named_steps['svc'].support_.shape[0]}")
plt.suptitle("Polynomial kernel: higher degree = more complex boundaries (often)", y=1.05)
plt.tight_layout()
plt.show()

## 6) Quick practice (optional)

Try these:
1. On the moons dataset, fix `gamma=0.5` and try `C` in `{0.1, 1, 10}`.  
   How does the boundary change?
2. Switch to `make_circles(...)` and compare kernels.
3. For polynomial kernel, vary `coef0` (e.g., 0, 1, 10). What happens?

**Hint:** you can reuse the plotting code above—just swap datasets and parameters.

## 7) Notes

- SVMs do **not** directly output probabilities (though you can enable probability calibration in sklearn).
- SVMs can be extended to multiclass classification via strategies like one-vs-one / one-vs-rest.
- Kernel SVMs can be slow on large datasets.

(Those points are summarized at the end of the slide deck.)